# Coordination candidates — u/CaseB

Finding the accounts most likely to be **coordinating** with u/CaseB, rather than merely
interacting with it a lot.

## Why two candidate sets

`network_case_CaseB_ego_network.ipynb` built 7,924 undirected alter–alter ties from replies inside
the ego's threads. Two different instruments pick out "close" accounts from that network:

| Set | Rule | What it rewards | Risk |
|---|---|---|---|
| **dense core** (203) | subgraph of highest average degree (Charikar peeling) | many ties to other alters | tie count tracks how much an account posts in r/The_Donald, so heavy regulars dominate |
| **significant** (128) | pairs sharing more posts than chance, given how many of the ego's threads each was present in (hypergeometric + BH, q<0.01) | improbably repeated pairings at any activity level | small counts on very light accounts |

They overlap by only 56 accounts. This notebook pulls the **union (275)** so neither framing
is discarded before the decisive test below.

## The decisive test

Co-presence in the ego's threads is expected — hundreds of accounts reply to each other in a
busy subreddit. The question that separates coordination from crowding is:

> **do these accounts keep replying to each other in threads the ego was never in?**

That needs each candidate's full history, which is what section 4 collects (~275 accounts,
roughly 2.3M items). Timing, lifecycle and content signals come later; the histories collected
here support all of them.

**Note on what the ties can and cannot show.** A tie found here is real. Ties these accounts
formed in threads outside this collection are not counted, so every count is a lower bound.

## 1. Imports and configuration

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import re
import glob
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from requests_ratelimiter import LimiterSession
import pyarrow as pa
import networkx as nx
from scipy.stats import hypergeom
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
# ---- configuration -------------------------------------------------------
EGO = 'CaseB'

Q_CUTOFF = 0.01                     # BH-corrected significance for a pair's shared posts
MAX_HISTORY_ITEMS = 250_000         # accounts above this are reported, not pulled
N_WORKERS = 4                       # combined request rate stays at the project's 5 req/s
REQUESTS_PER_SECOND = 5

COMMENT_FIELDS = ['id', 'author', 'parent_id', 'link_id', 'created_utc', 'subreddit']
POST_FIELDS    = ['id', 'author', 'created_utc', 'subreddit', 'num_comments']

EGO_DIR   = f'ego_net_{EGO}_output'          # outputs of the ego-network notebook
OUT_DIR   = f'coordination_{EGO}_output'
CACHE_DIR = os.path.join(OUT_DIR, 'candidate_histories')
FIG_DIR   = os.path.join('..', 'p2_figures')
for d in (OUT_DIR, CACHE_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

print(f'ego = {EGO}\nreading  <- {EGO_DIR}\nwriting  -> {OUT_DIR}')

## 2. API layer

In [ ]:
# Arctic Shift API helpers, safe to call from several threads at once.
BASE = 'https://arctic-shift.photon-reddit.com/api'
# One rate-limited HTTP session per worker thread; the total rate is shared across N_WORKERS.
_local = threading.local()


def get_session():
    if not hasattr(_local, 'session'):
        _local.session = LimiterSession(per_minute=int(60 * REQUESTS_PER_SECOND / N_WORKERS))
        _local.session.headers['User-Agent'] = 'dissertation-ego-network/1.0 (research)'
    return _local.session


class BadQuery(Exception):
    """HTTP 400 - a malformed query that retrying cannot fix."""


# One GET with up to 10 retries and growing waits; a 400 (bad query) is raised at once.
def arctic_get(path, **params):
    retry_count = 0
    last_error = None
    while retry_count < 10:
        try:
            response = get_session().get(f'{BASE}/{path}', params=params, timeout=120)
            if response.status_code == 200:
                return response.json().get('data') or []
            if response.status_code == 400:
                raise BadQuery(f'400 from {path} {params}: {response.text[:200]}')
            last_error = f'HTTP {response.status_code}'
        except BadQuery:
            raise
        except Exception as e:
            last_error = f'{e.__class__.__name__}: {e}'
        retry_count += 1
        print(f'  retry {retry_count} for {path} {params}: {last_error}')
        time.sleep(min(2 ** retry_count, 60))
    raise RuntimeError(f'Failed {path} {params} after 10 retries. Last: {last_error}')


def arctic_paginate(path, limit='auto', **params):
    """`after` is exclusive, so the cursor sits one second before the last row seen and rows
    are deduplicated on id - nothing posted in the same second as a page boundary is lost."""
    rows, seen = [], set()
    after = None
    while True:
        page_params = dict(params, limit=limit, sort='asc')
        if after is not None:
            page_params['after'] = after
        data = arctic_get(path, **page_params)
        new = [r for r in data if r['id'] not in seen]
        seen.update(r['id'] for r in new)
        rows.extend(new)
        if len(data) < 100:
            break
        last = data[-1]['created_utc']
        if not new:
            print(f'  WARNING cursor stalled at {last} for {path} {params}; stepping past it')
            after = last
        else:
            after = last - 1
    return rows


print('API layer ready')

## 3. The candidate set

Rebuilt from the ego-network notebook's saved outputs, so the two sets are reproducible here
rather than copied in by hand.

In [ ]:
# Rebuild the two candidate sets from the ego-network outputs: alter pairs whose shared threads beat chance,
# and the dense core of the alter-alter network.
alters = pd.read_parquet(os.path.join(EGO_DIR, f'alters_min2_{EGO}.parquet'))
ties = pd.read_parquet(os.path.join(EGO_DIR, f'alter_ties_{EGO}.parquet'))
dense = pd.read_parquet(os.path.join(EGO_DIR, f'dense_core_{EGO}.parquet'))
thread_posts = pd.read_parquet(os.path.join(EGO_DIR, 'thread_post_authors.parquet'))
threads = (pd.concat([pd.read_parquet(f) for f in
                      sorted(glob.glob(os.path.join(EGO_DIR, 'thread_cache', 'threads_*.parquet')))],
                     ignore_index=True).drop_duplicates('id'))
kept = set(alters['alter'])
print(f'retained alters {len(kept):,} · ties {len(ties):,} · dense core {len(dense):,} · '
      f'thread comments {len(threads):,}')

# --- presence: threads where an alter commented OR authored the submission ---------------
# (a tie can come from replying to an alter's submission, so comment presence alone
#  understates exposure and would inflate significance)
present = pd.concat([
    threads.loc[threads['author'].isin(kept), ['author', 'link_id']],
    (thread_posts[thread_posts['author'].isin(kept)]
     .assign(link_id=lambda d: 't3_' + d['id'])[['author', 'link_id']]),
]).drop_duplicates()
threads_present = present.groupby('author')['link_id'].nunique()
n_threads = threads['link_id'].nunique()

# Hypergeometric test per tie: shared threads given how many threads each alter was present in;
# Benjamini-Hochberg q-values across all ties.
ka = ties['a'].map(threads_present).to_numpy()
kb = ties['b'].map(threads_present).to_numpy()
obs = ties['posts'].to_numpy()
assert (obs <= np.minimum(ka, kb)).all(), 'a pair shares more posts than either was present in'

ties = ties.assign(threads_a=ka, threads_b=kb, expected=ka * kb / n_threads,
                   pval=hypergeom.sf(obs - 1, n_threads, ka, kb)).sort_values('pval')
m = len(ties)
ties['qval'] = np.minimum.accumulate((ties['pval'] * m / np.arange(1, m + 1))[::-1])[::-1]
significant = ties[ties['qval'] < Q_CUTOFF]
sig_alters = set(significant['a']) | set(significant['b'])
dense_alters = set(dense['alter'])

print(f'\nties beating chance (BH q<{Q_CUTOFF}): {len(significant):,} of {m:,} '
      f'({len(significant) / m:.1%})')
print(f'  median observed {significant["posts"].median():.0f} shared posts vs '
      f'{significant["expected"].median():.3f} expected')
print(f'significant alters {len(sig_alters)} · dense-core alters {len(dense_alters)} · '
      f'overlap {len(sig_alters & dense_alters)}')

# Candidate union, labelled by which set(s) each account came from; saved.
source = {a: ('both' if a in sig_alters and a in dense_alters
              else 'significant' if a in sig_alters else 'dense') for a in sig_alters | dense_alters}
candidates = (pd.DataFrame({'account': sorted(source), 'set': [source[a] for a in sorted(source)]})
              .merge(alters[['alter', 'contacts', 'replies', 'account_status']]
                     .rename(columns={'alter': 'account'}), on='account', how='left')
              .merge(dense[['alter', 'ties_in_core', 'core_shell', 'alter_ties']]
                     .rename(columns={'alter': 'account'}), on='account', how='left'))
candidates['threads_present'] = candidates['account'].map(threads_present)
candidates = candidates.sort_values(['set', 'account']).reset_index(drop=True)
candidates.to_parquet(os.path.join(OUT_DIR, f'candidates_{EGO}.parquet'),
                      engine='pyarrow', compression='zstd')

print(f'\ncandidate union: {len(candidates)} accounts')
print(candidates.groupby('set').agg(n=('account', 'size'),
                                    median_ties=('alter_ties', 'median'),
                                    median_threads=('threads_present', 'median'),
                                    suspended=('account_status', lambda s: f'{(s == "Suspended").mean():.0%}'),
                                    deleted=('account_status', lambda s: f'{(s == "Deleted").mean():.0%}')).to_string())

## 4. Pull each candidate's full history

Cached per account, so an interrupted run resumes. Expected volume is checked first and
anything over `MAX_HISTORY_ITEMS` is reported rather than pulled.

In [ ]:
# Before pulling histories: each candidate's archive totals (cached), and which accounts are too large to pull
# (over MAX_HISTORY_ITEMS).
cache_path = lambda name, kind: os.path.join(CACHE_DIR, f'{name}_{kind}.parquet')

meta_path = os.path.join(OUT_DIR, 'candidate_meta.parquet')
if os.path.exists(meta_path):
    cand_meta = pd.read_parquet(meta_path)
else:
    rows = []
    for name in candidates['account']:
        hits = [u for u in arctic_get('users/search', author=name)
                if u['author'].lower() == name.lower()]
        meta = hits[0].get('_meta', {}) if hits else {}
        rows.append({'account': name,
                     'num_comments': meta.get('num_comments'),
                     'num_posts': meta.get('num_posts'),
                     'total_karma': meta.get('total_karma'),
                     'earliest_comment_at': meta.get('earliest_comment_at'),
                     'last_comment_at': meta.get('last_comment_at')})
    cand_meta = pd.DataFrame(rows)
    cand_meta.to_parquet(meta_path, engine='pyarrow')
cand_meta['expected_items'] = cand_meta[['num_comments', 'num_posts']].fillna(0).sum(axis=1)

too_big = cand_meta.loc[cand_meta['expected_items'] > MAX_HISTORY_ITEMS, 'account'].tolist()
to_pull = [a for a in candidates['account'] if a not in too_big]
expected = cand_meta.set_index('account')['expected_items']
print(f'accounts to pull: {len(to_pull)}  (skipped as too large: {too_big})')
print(f'expected items: {int(expected.reindex(to_pull).sum()):,}')
print(f'no archive record: {cand_meta.loc[cand_meta["expected_items"] == 0, "account"].tolist()}')

In [ ]:
def pull_history(name):
    """Pull and cache one account's comments and posts. Written to a temp file and renamed,
    so an interrupted run never leaves a partial cache file behind."""
    for kind, fields in (('comments', COMMENT_FIELDS), ('posts', POST_FIELDS)):
        path = cache_path(name, kind)
        if os.path.exists(path):
            continue
        rows = arctic_paginate(f'{kind}/search', author=name, fields=','.join(fields))
        df = pd.DataFrame(rows, columns=fields)
        df = df[df['author'].str.lower() == name.lower()]     # author search is case-insensitive
        df.to_parquet(path + '.tmp', engine='pyarrow', compression='zstd')
        os.replace(path + '.tmp', path)
    return name


pending = [a for a in to_pull
           if not (os.path.exists(cache_path(a, 'comments')) and os.path.exists(cache_path(a, 'posts')))]
total_items = expected.reindex(pending).sum()
print(f'{len(to_pull) - len(pending)} already cached, {len(pending)} to pull '
      f'(~{int(total_items):,} items) with {N_WORKERS} workers')

t_start, done_items = time.time(), 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    # largest first, so the long pulls run side by side rather than queueing at the end
    futures = [pool.submit(pull_history, a)
               for a in expected.reindex(pending).sort_values(ascending=False).index]
    for i, fut in enumerate(as_completed(futures), 1):
        done_items += expected.get(fut.result(), 0)
        if i % 10 == 0 or i == len(futures):
            print(f'  {i}/{len(futures)} accounts  {done_items / max(total_items, 1):.0%} of items  '
                  f'{(time.time() - t_start) / 60:.1f} min', flush=True)

In [ ]:
cand_comments = pd.concat([pd.read_parquet(cache_path(a, 'comments')) for a in to_pull], ignore_index=True)
cand_posts = pd.concat([pd.read_parquet(cache_path(a, 'posts')) for a in to_pull], ignore_index=True)
print(f'candidate comments: {len(cand_comments):,}   posts: {len(cand_posts):,}')

# pulled vs the archive's own counts - a shortfall would mean pagination lost rows
got = (cand_comments.groupby('author').size().rename('got_comments').to_frame()
       .join(cand_posts.groupby('author').size().rename('got_posts'), how='outer'))
coverage = (cand_meta.set_index('account').reindex(to_pull)[['num_comments', 'num_posts']]
            .join(got).fillna(0))
coverage['comment_ratio'] = coverage['got_comments'] / coverage['num_comments'].replace(0, np.nan)
print(f'comments pulled / archive count: median {coverage["comment_ratio"].median():.3f}, '
      f'min {coverage["comment_ratio"].min():.3f}')
coverage.sort_values('comment_ratio').head(5)

## 5. Do the candidates interact away from the ego?

Every reply between two candidates is classified by whether the ego was in that thread:

- **in the ego's threads** — already known from the ego-network notebook; expected from
  co-presence alone.
- **away from the ego** — the ego never posted or commented in that thread. This is the
  signal that separates a real relationship from crowding in the same subreddit.

In [ ]:
# every piece of content the candidates wrote, by fullname
content_author = pd.concat([
    pd.Series(cand_comments['author'].values, index='t1_' + cand_comments['id']),
    pd.Series(cand_posts['author'].values, index='t3_' + cand_posts['id']),
])
content_author = content_author[~content_author.index.duplicated()]

replies = cand_comments.assign(parent_author=cand_comments['parent_id'].map(content_author))
replies = replies[replies['parent_author'].notna()
                  & (replies['parent_author'].str.lower() != replies['author'].str.lower())]

ego_threads = set(threads['link_id'])                      # threads the ego took part in
replies = replies.assign(with_ego=replies['link_id'].isin(ego_threads))

u = np.minimum(replies['author'], replies['parent_author'])
v = np.maximum(replies['author'], replies['parent_author'])
paired = replies.assign(a=u, b=v)
pair_ties = (paired.groupby(['a', 'b'])
             .agg(posts=('link_id', 'nunique'), replies=('id', 'size'),
                  first_utc=('created_utc', 'min'), last_utc=('created_utc', 'max'),
                  subreddits=('subreddit', lambda s: ','.join(sorted(set(s))[:5])))
             .reset_index())
# count each pair's posts inside / away from the ego's threads with explicit joins rather
# than index tricks inside the aggregation
with_ego = (paired[paired['with_ego']].groupby(['a', 'b'])['link_id'].nunique()
            .rename('posts_with_ego').reset_index())
away_cnt = (paired[~paired['with_ego']].groupby(['a', 'b'])['link_id'].nunique()
            .rename('posts_away').reset_index())
pair_ties = (pair_ties.merge(with_ego, on=['a', 'b'], how='left')
                      .merge(away_cnt, on=['a', 'b'], how='left'))
pair_ties[['posts_with_ego', 'posts_away']] = pair_ties[['posts_with_ego', 'posts_away']].fillna(0).astype(int)
assert (pair_ties['posts_with_ego'] + pair_ties['posts_away'] == pair_ties['posts']).all()
pair_ties = pair_ties.sort_values(['posts_away', 'posts'], ascending=False).reset_index(drop=True)
pair_ties.to_parquet(os.path.join(OUT_DIR, f'candidate_pair_ties_{EGO}.parquet'),
                     engine='pyarrow', compression='zstd')

away = pair_ties[pair_ties['posts_away'] > 0]
print(f'candidate pairs replying to each other at all      : {len(pair_ties):,}')
print(f'  pairs that also do it away from the ego          : {len(away):,} ({len(away) / max(len(pair_ties), 1):.0%})')
print(f'  posts away from the ego                          : {int(pair_ties["posts_away"].sum()):,}')
print(f'  accounts involved in an away-from-ego tie        : '
      f'{len(set(away["a"]) | set(away["b"])):,} of {len(to_pull)}')
away.head(25)

## 6. Chance-correcting the off-ego ties

Section 5 showed 98% of candidate pairs interact away from the ego — a test that flags
everyone flags no one. The reason is co-membership: 18,789 of the tie-subreddit mentions are
r/The_Donald, so heavy users of one subreddit inevitably meet each other. Three nulls are
applied in increasing strictness, each comparing a pair's **observed shared posts** against
an expectation built from their own activity.

1. **subreddit activity** — per subreddit, expectation `k_a · k_b / N_s` where `k` is posts the
   account touched in that subreddit and `N_s` is posts any candidate touched there.
2. **+ thread popularity** — the bipartite configuration model,
   `λ = k_a · k_b · Σ n_p² / T²`, where `n_p` is how many candidates a post drew. Big threads
   pull everyone in, and this removes that.
3. **conversation only, calibrated** — comment→comment replies only (dropping replies to an
   account's *submissions*, which are audience, not conversation), with the co-presence
   expectation rescaled by the global conversation rate so like is compared with like.

In [ ]:
from scipy.stats import poisson

pairs_sub = (paired.groupby(['a', 'b', 'subreddit'])['link_id'].nunique()
             .rename('posts').reset_index())
touch = pd.concat([cand_comments[['author', 'subreddit', 'link_id']],
                   cand_posts.assign(link_id='t3_' + cand_posts['id'])[['author', 'subreddit', 'link_id']]
                   ]).drop_duplicates()
k_as = touch.groupby(['author', 'subreddit'])['link_id'].nunique().rename('k')
n_p = touch.groupby(['subreddit', 'link_id'])['author'].nunique().rename('n_p')
sub_stat = n_p.groupby('subreddit').agg(T='sum', S2=lambda s: (s ** 2).sum())


def expectations(obs_table):
    # Attach both nulls to a pair x subreddit table of observed shared posts.
    m = (obs_table
         .merge(k_as.rename('k_a').reset_index().rename(columns={'author': 'a'}), on=['a', 'subreddit'])
         .merge(k_as.rename('k_b').reset_index().rename(columns={'author': 'b'}), on=['b', 'subreddit'])
         .join(sub_stat, on='subreddit'))
    m['lam_activity'] = m['k_a'] * m['k_b'] / m['T']          # N_s == T: posts touched in the subreddit
    m['lam_popularity'] = m['k_a'] * m['k_b'] * m['S2'] / m['T'] ** 2
    return (m.groupby(['a', 'b'])
            .agg(posts=('posts', 'sum'), lam_activity=('lam_activity', 'sum'),
                 lam_popularity=('lam_popularity', 'sum'), subreddits=('subreddit', 'nunique'))
            .reset_index())


def bh(frame, lam_col, alpha=0.01):
    f = frame.assign(pval=poisson.sf(frame['posts'] - 1, frame[lam_col])).sort_values('pval')
    n = len(f)
    f['qval'] = np.minimum.accumulate((f['pval'] * n / np.arange(1, n + 1))[::-1])[::-1]
    return f, f[f['qval'] < alpha]


all_pairs = expectations(pairs_sub)
_, sig_activity = bh(all_pairs, 'lam_activity')
_, sig_popularity = bh(all_pairs, 'lam_popularity')
print(f'pairs tested: {len(all_pairs):,}')
print(f'  null 1 (subreddit activity)      -> {len(sig_activity):,} significant pairs, '
      f'{len(set(sig_activity["a"]) | set(sig_activity["b"]))} accounts')
print(f'  null 2 (+ thread popularity)     -> {len(sig_popularity):,} significant pairs, '
      f'{len(set(sig_popularity["a"]) | set(sig_popularity["b"]))} accounts')

# what are the null-2 survivors actually made of?
hub_check = paired[paired['a'].isin(sig_popularity['a']) | paired['b'].isin(sig_popularity['b'])]
print(f'  of those replies, share whose parent is a submission rather than a comment: '
      f'{hub_check["parent_id"].str.startswith("t3_").mean():.0%}')

In [ ]:
# --- null 3: conversation only, calibrated ------------------------------------------------
conv = paired[paired['parent_id'].str.startswith('t1_')]
conv_sub = (conv.groupby(['a', 'b', 'subreddit'])['link_id'].nunique().rename('posts').reset_index())
conv_pairs = expectations(conv_sub)
# co-presence expectation overstates conversation, so rescale by the observed global rate
rho = conv_pairs['posts'].sum() / conv_pairs['lam_popularity'].sum()
conv_pairs['lam_conversation'] = conv_pairs['lam_popularity'] * rho
conv_all, conv_sig = bh(conv_pairs, 'lam_conversation')
print(f'comment-to-comment replies: {len(conv):,} of {len(paired):,} candidate-candidate replies')
print(f'calibration rho = {rho:.3f} conversations per unit co-presence')
print(f'null 3 -> {len(conv_sig):,} significant pairs, '
      f'{len(set(conv_sig["a"]) | set(conv_sig["b"]))} accounts')

CG = nx.Graph()
CG.add_weighted_edges_from(conv_sig[['a', 'b', 'posts']].itertuples(index=False, name=None))
comps = sorted(nx.connected_components(CG), key=len, reverse=True)
status = candidates.set_index('account')['account_status']
print(f'network: {CG.number_of_nodes()} accounts, {CG.number_of_edges()} pairs, '
      f'components {len(comps)} sizes {[len(c) for c in comps[:6]]}')
print('status of that network:',
      status.reindex(sorted(CG.nodes())).value_counts().to_dict(),
      '| suspended share %.0f%% vs %.0f%% across all candidates'
      % (100 * status.reindex(sorted(CG.nodes())).eq('Suspended').mean(), 100 * status.eq('Suspended').mean()))
conv_all.to_parquet(os.path.join(OUT_DIR, f'conversation_pairs_{EGO}.parquet'),
                    engine='pyarrow', compression='zstd')
nx.write_graphml(CG, os.path.join(OUT_DIR, f'conversation_significant_{EGO}.graphml'))
conv_sig.head(20)

## 7. The general alter–alter tie network, and its densest part

Section 11 of the ego notebook kept the densest subgraph of ties formed **inside the ego's
threads**. This is the same construction over the **general** network — every reply between
two candidates anywhere on Reddit — saved as a graph for later use.

In [ ]:
def densest_subgraph(graph):
    # Charikar greedy peeling: drop the lowest-degree node repeatedly and keep the best
    # average-degree subgraph seen.
    H = graph.copy()
    best = (list(H.nodes()), H.number_of_edges() / max(H.number_of_nodes(), 1))
    while H.number_of_nodes() > 2:
        n = min(H.degree(), key=lambda kv: kv[1])[0]
        H.remove_node(n)
        d = H.number_of_edges() / H.number_of_nodes()
        if d > best[1]:
            best = (list(H.nodes()), d)
    return best


GEN = nx.Graph()
for _, r in pair_ties.iterrows():
    GEN.add_edge(r['a'], r['b'], weight=int(r['posts']), replies=int(r['replies']),
                 posts_away=int(r['posts_away']))
dense_nodes, _ = densest_subgraph(GEN)
GEN_D = GEN.subgraph(sorted(dense_nodes)).copy()
for n in GEN.nodes():
    GEN.nodes[n]['account_status'] = status.get(n, 'not checked')
    GEN.nodes[n]['candidate_set'] = candidates.set_index('account')['set'].get(n, '')
for n in GEN_D.nodes():
    GEN_D.nodes[n]['account_status'] = status.get(n, 'not checked')

print(f'general tie network : {GEN.number_of_nodes()} accounts, {GEN.number_of_edges():,} ties, '
      f'avg degree {2 * GEN.number_of_edges() / GEN.number_of_nodes():.1f}, density {nx.density(GEN):.3f}')
print(f'densest subgraph    : {GEN_D.number_of_nodes()} accounts, {GEN_D.number_of_edges():,} ties, '
      f'avg degree {2 * GEN_D.number_of_edges() / GEN_D.number_of_nodes():.1f}, density {nx.density(GEN_D):.3f}')
print('status in the densest subgraph:',
      status.reindex(sorted(GEN_D.nodes())).value_counts().to_dict())
nx.write_graphml(GEN, os.path.join(OUT_DIR, f'general_tie_network_{EGO}.graphml'))
nx.write_graphml(GEN_D, os.path.join(OUT_DIR, f'general_tie_network_densest_{EGO}.graphml'))
print(f'saved general_tie_network_{EGO}.graphml and general_tie_network_densest_{EGO}.graphml')

## 8. Subreddit diversity of ties

How many **distinct subreddits** a relationship spans, against how many posts it covers. A
tie concentrated in one subreddit is ordinary co-membership; a tie that follows the other
account across many different subreddits is a relationship between the accounts themselves.

Measured for ego ↔ alter contacts and for alter ↔ alter ties, then tested against account
outcome.

In [ ]:
from scipy.stats import mannwhitneyu, fisher_exact

# --- ego <-> alter -------------------------------------------------------------------
ego_int = pd.read_parquet(os.path.join(EGO_DIR, f'interactions_{EGO}.parquet'))
ego_div = (ego_int[ego_int['alter'].isin(kept)]
           .groupby('alter')
           .agg(posts=('link_id', 'nunique'), replies=('id', 'size'),
                subreddits=('subreddit', 'nunique'),
                subreddit_list=('subreddit', lambda s: ','.join(sorted(set(s))[:8])))
           .reset_index()
           .merge(alters[['alter', 'account_status']], on='alter'))
ego_div['subs_per_post'] = ego_div['subreddits'] / ego_div['posts']
ego_div = ego_div.sort_values(['subreddits', 'posts'], ascending=False).reset_index(drop=True)
ego_div.to_parquet(os.path.join(OUT_DIR, f'ego_alter_diversity_{EGO}.parquet'),
                   engine='pyarrow', compression='zstd')

print('ego <-> alter: alters by number of distinct subreddits shared with the ego')
display(ego_div.head(12)[['alter', 'posts', 'replies', 'subreddits', 'subs_per_post',
                          'account_status', 'subreddit_list']])
print('alters by subreddit span:', ego_div['subreddits'].value_counts().sort_index().to_dict())
broad = ego_div['subreddits'] >= 3
print(f'alters spanning >=3 subreddits with the ego: {broad.sum()} · '
      f'suspended {100 * ego_div.loc[broad, "account_status"].eq("Suspended").mean():.0f}% '
      f'vs {100 * ego_div["account_status"].eq("Suspended").mean():.0f}% baseline · '
      f'Fisher p={fisher_exact(pd.crosstab(broad, ego_div["account_status"] == "Suspended").values)[1]:.3f}')

In [ ]:
# --- alter <-> alter -----------------------------------------------------------------
pair_div = (pairs_sub.groupby(['a', 'b'])
            .agg(posts=('posts', 'sum'), subreddits=('subreddit', 'nunique'),
                 subreddit_list=('subreddit', lambda s: ','.join(sorted(s)[:8])))
            .reset_index())
pair_div['subs_per_post'] = pair_div['subreddits'] / pair_div['posts']
pair_div = pair_div.sort_values(['subreddits', 'posts'], ascending=False).reset_index(drop=True)
pair_div.to_parquet(os.path.join(OUT_DIR, f'alter_alter_diversity_{EGO}.parquet'),
                    engine='pyarrow', compression='zstd')
print('alter <-> alter: pairs spanning the most subreddits')
display(pair_div.head(12))
print('pairs by subreddit span:', pair_div['subreddits'].value_counts().sort_index().head(8).to_dict())

# per-account breadth across all of its candidate ties
breadth = (pd.concat([pairs_sub.rename(columns={'a': 'account', 'b': 'other'}),
                      pairs_sub.rename(columns={'b': 'account', 'a': 'other'})])
           .groupby('account')
           .agg(partners=('other', 'nunique'), tie_subreddits=('subreddit', 'nunique'),
                tie_posts=('posts', 'sum'))
           .reset_index()
           .merge(candidates[['account', 'set', 'account_status', 'contacts']], on='account'))
breadth['subs_per_partner'] = breadth['tie_subreddits'] / breadth['partners']
breadth = breadth.sort_values('tie_subreddits', ascending=False).reset_index(drop=True)
breadth.to_parquet(os.path.join(OUT_DIR, f'tie_breadth_{EGO}.parquet'),
                   engine='pyarrow', compression='zstd')
print('\naccounts whose ties span the most subreddits:')
display(breadth.head(15)[['account', 'partners', 'tie_subreddits', 'tie_posts', 'contacts',
                          'account_status', 'set']])

# does breadth track enforcement, and is it just activity?
for n in (10, 20, 40):
    t = breadth.head(n)
    print(f'top {n:>2} by breadth: suspended {100 * t["account_status"].eq("Suspended").mean():>3.0f}% · '
          f'suspended-or-deleted {100 * t["account_status"].isin(["Suspended", "Deleted"]).mean():>3.0f}%')
print(f'all candidates  : suspended {100 * breadth["account_status"].eq("Suspended").mean():>3.0f}% · '
      f'suspended-or-deleted {100 * breadth["account_status"].isin(["Suspended", "Deleted"]).mean():>3.0f}%')
print('\nmedian by status:')
print(breadth.groupby('account_status')[['tie_subreddits', 'partners', 'tie_posts', 'subs_per_partner']]
      .median().to_string())
susp = breadth.loc[breadth['account_status'] == 'Suspended', 'tie_subreddits']
act = breadth.loc[breadth['account_status'] == 'Active', 'tie_subreddits']
print(f'\nMann-Whitney breadth suspended > active: p={mannwhitneyu(susp, act, alternative="greater")[1]:.4g}')
print('normalised (subreddits per partner), suspended > active: p=%.4g' % mannwhitneyu(
    breadth.loc[breadth['account_status'] == 'Suspended', 'subs_per_partner'],
    breadth.loc[breadth['account_status'] == 'Active', 'subs_per_partner'], alternative='greater')[1])

## 9. Save outputs

In [ ]:
# Add each candidate's ties away from the ego (partners and shared posts), save candidates and significant ties, list outputs.
candidates_out = candidates.copy()
away_deg = (pd.concat([away[['a', 'posts_away']].rename(columns={'a': 'account'}),
                       away[['b', 'posts_away']].rename(columns={'b': 'account'})])
            .groupby('account').agg(away_partners=('posts_away', 'size'),
                                    away_posts=('posts_away', 'sum')))
candidates_out = candidates_out.merge(away_deg, left_on='account', right_index=True, how='left')
candidates_out[['away_partners', 'away_posts']] = candidates_out[['away_partners', 'away_posts']].fillna(0).astype(int)
candidates_out = candidates_out.sort_values(['away_posts', 'away_partners'], ascending=False).reset_index(drop=True)
candidates_out.to_parquet(os.path.join(OUT_DIR, f'candidates_{EGO}.parquet'),
                          engine='pyarrow', compression='zstd')
significant.to_parquet(os.path.join(OUT_DIR, f'significant_ties_{EGO}.parquet'),
                       engine='pyarrow', compression='zstd')

print('candidates ranked by interaction away from the ego:')
display(candidates_out.head(25)[['account', 'set', 'account_status', 'contacts', 'alter_ties',
                                 'threads_present', 'away_partners', 'away_posts']])
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    if os.path.isfile(p):
        print(f'  {p}  ({os.path.getsize(p) / 1024:.1f} KB)')